# Walmart Demand Forecasting
## Notebook 04: Forecasting Model Development
## Objectives
Develop and compare forecasting models to predict Walmart's weekly sales.
### Dataset
- walmart_features.csv
### Tasks
- Load the processed feature dataset.
- Split the data using a time-based validation strategy.
- Establish baseline forecasting models.
- Train and compare machine learning models.
- Analyze feature importance.
- Generate weekly sales predictions.
### Expected Output
By the end of this notebook, we will have one or more forecasting models capable of predicting weekly sales. Model performance will be compared against baseline approaches to determine whether machine learning provides meaningful improvements in forecasting accuracy.

#1. Load and check data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(
    PROCESSED_PATH / "walmart_features.csv",
    parse_dates=["Date"]
)
df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,...,Type,Size,Year,Month,Week,lag_1,lag_2,lag_4,rolling_mean_4,rolling_std_4
0,1,1,2011-11-11,18689.54,59.11,3.297,10382.90,6115.67,215.07,2406.62,...,A,151315,2011,11,45,39886.06,31579.90,23077.55,29473.8275,7984.186737
1,1,1,2011-11-18,19050.66,62.25,3.308,6074.12,254.39,51.98,427.39,...,A,151315,2011,11,46,18689.54,39886.06,23351.80,28376.8250,9341.958158
2,1,1,2011-11-25,20911.25,60.14,3.236,410.31,98.00,55805.51,8.00,...,A,151315,2011,11,47,19050.66,18689.54,31579.90,27301.5400,10310.481280
3,1,1,2011-12-02,25293.49,48.91,3.172,5629.51,68.00,1398.11,2084.64,...,A,151315,2011,12,48,20911.25,19050.66,39886.06,24634.3775,10214.279082
4,1,1,2011-12-09,33305.92,43.93,3.158,4640.65,19.00,105.02,3639.42,...,A,151315,2011,12,49,25293.49,20911.25,18689.54,20986.2350,3032.014011


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96905 entries, 0 to 96904
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Store           96905 non-null  int64         
 1   Dept            96905 non-null  int64         
 2   Date            96905 non-null  datetime64[us]
 3   Weekly_Sales    96905 non-null  float64       
 4   Temperature     96905 non-null  float64       
 5   Fuel_Price      96905 non-null  float64       
 6   MarkDown1       96905 non-null  float64       
 7   MarkDown2       96905 non-null  float64       
 8   MarkDown3       96905 non-null  float64       
 9   MarkDown4       96905 non-null  float64       
 10  MarkDown5       96905 non-null  float64       
 11  CPI             96905 non-null  float64       
 12  Unemployment    96905 non-null  float64       
 13  IsHoliday       96905 non-null  int64         
 14  Type            96905 non-null  str           
 15  Size         

In [4]:
print(df.shape)

(96905, 24)


#2. Select Features and Target

In [6]:
target = "Weekly_Sales"
X = df.drop(columns=[target, 'Date'])
y = df[target]
print(X.shape)
print(y.shape)
#Check feature types
X.dtypes

(96905, 22)
(96905,)


Store               int64
Dept                int64
Temperature       float64
Fuel_Price        float64
MarkDown1         float64
MarkDown2         float64
MarkDown3         float64
MarkDown4         float64
MarkDown5         float64
CPI               float64
Unemployment      float64
IsHoliday           int64
Type                  str
Size                int64
Year                int64
Month               int64
Week                int64
lag_1             float64
lag_2             float64
lag_4             float64
rolling_mean_4    float64
rolling_std_4     float64
dtype: object

In [8]:
#Identify Numerical vs Categorical Features
numerical_features = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment",
    "Size",
    "Year",
    "Month",
    "Week",
    "lag_1",
    "lag_2",
    "lag_4",
    "rolling_mean_4",
    "rolling_std_4"
]

# Categorical features
categorical_features = [
    "Store",
    "Dept",
    "Type",
    "IsHoliday"
]

Numerical Features:
Index(['Store', 'Dept', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
       'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment',
       'IsHoliday', 'Size', 'Year', 'Month', 'Week', 'lag_1', 'lag_2', 'lag_4',
       'rolling_mean_4', 'rolling_std_4'],
      dtype='str')

Categorical Features:
Index(['Type'], dtype='str')


#3. Train/test split

In [9]:
df["Week"] = df["Date"].dt.isocalendar().week

,Date,Year,Month,Week
0,2011-11-11,2011,11,45
1,2011-11-18,2011,11,46
2,2011-11-25,2011,11,47
3,2011-12-02,2011,12,48
4,2011-12-09,2011,12,49
5,2011-12-16,2011,12,50
6,2011-12-23,2011,12,51
7,2011-12-30,2011,12,52
8,2012-01-06,2012,1,1
9,2012-01-13,2012,1,2
